Cell 1: MLflow observability setup

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 28 — MLflow Agent Monitoring + Evaluation
# Cell 1 — MLflow setup, experiment config, table checks
# ─────────────────────────────────────────────────────────────

import json
import platform
from datetime import datetime, timezone

import mlflow
from pyspark.sql import functions as F

# ─────────────────────────────────────────────────────────────
# Project config
# ─────────────────────────────────────────────────────────────

GOLD_SCHEMA = "supplysage_gold"

# Core agent tables from earlier notebooks
CHAT_CONTEXT_TABLE = f"{GOLD_SCHEMA}.gold_chat_context_snapshots"
EMBEDDINGS_TABLE = f"{GOLD_SCHEMA}.gold_rag_embeddings"
RETRIEVAL_INDEX_TABLE = f"{GOLD_SCHEMA}.gold_rag_retrieval_index"
AGENT_RUN_LOG_TABLE = f"{GOLD_SCHEMA}.gold_supplier_risk_agent_run_logs"

# New Notebook 28 tables
AGENT_EVAL_TABLE = f"{GOLD_SCHEMA}.gold_supplier_risk_agent_eval_results"
AGENT_MONITORING_TABLE = f"{GOLD_SCHEMA}.gold_supplier_risk_agent_monitoring_metrics"

# MLflow experiment
# Use a workspace path so runs are easy to find in Databricks Experiments.
MLFLOW_EXPERIMENT_NAME = "/Users/vigya.awasthi@tamu.edu/supplysage_ai_agent_monitoring"

# Agent metadata
AGENT_NAME = "supplysage_supplier_risk_agent"
AGENT_VERSION = "v1_manual_graph"
PROJECT_NAME = "SupplySage AI"

# Evaluation defaults
EVAL_QUESTION_SET_VERSION = "v1"
DEFAULT_TOP_K_FINAL = 8

# ─────────────────────────────────────────────────────────────
# MLflow setup
# ─────────────────────────────────────────────────────────────

try:
    mlflow.set_tracking_uri("databricks")
except Exception as e:
    print("Could not explicitly set Databricks tracking URI. Continuing with default tracking URI.")
    print("Tracking URI warning:", str(e))

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)

print("MLflow experiment configured.")
print("Experiment name:", MLFLOW_EXPERIMENT_NAME)
print("Experiment ID:", experiment.experiment_id if experiment else None)
print("Tracking URI:", mlflow.get_tracking_uri())

# ─────────────────────────────────────────────────────────────
# Environment metadata
# ─────────────────────────────────────────────────────────────

env_metadata = {
    "project_name": PROJECT_NAME,
    "agent_name": AGENT_NAME,
    "agent_version": AGENT_VERSION,
    "gold_schema": GOLD_SCHEMA,
    "python_version": platform.python_version(),
    "mlflow_version": mlflow.__version__,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "has_mlflow_trace": hasattr(mlflow, "trace")
}

print("\nEnvironment metadata:")
print(json.dumps(env_metadata, indent=2))

# ─────────────────────────────────────────────────────────────
# Validate required upstream tables
# ─────────────────────────────────────────────────────────────

required_tables = [
    CHAT_CONTEXT_TABLE,
    EMBEDDINGS_TABLE,
    RETRIEVAL_INDEX_TABLE,
    AGENT_RUN_LOG_TABLE
]

table_counts = {}

for table_name in required_tables:
    print(f"\nChecking table: {table_name}")

    try:
        row_count = spark.table(table_name).count()
        table_counts[table_name] = row_count
        print(f"Rows: {row_count}")
    except Exception as e:
        raise ValueError(f"Required table check failed for {table_name}: {str(e)}")

assert table_counts[CHAT_CONTEXT_TABLE] > 0, "No supplier snapshots found."
assert table_counts[EMBEDDINGS_TABLE] > 0, "No RAG embeddings found."
assert table_counts[RETRIEVAL_INDEX_TABLE] > 0, "No retrieval index rows found."
assert table_counts[AGENT_RUN_LOG_TABLE] > 0, "No prior agent run logs found."

print("\nAll required tables are available.")
print("Notebook 28 setup complete.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 28 — Cell 2A
# Restore config variables after notebook state reset
# ─────────────────────────────────────────────────────────────

import json
import platform
from datetime import datetime, timezone

import mlflow
from pyspark.sql import functions as F

GOLD_SCHEMA = "supplysage_gold"

CHAT_CONTEXT_TABLE = f"{GOLD_SCHEMA}.gold_chat_context_snapshots"
EMBEDDINGS_TABLE = f"{GOLD_SCHEMA}.gold_rag_embeddings"
RETRIEVAL_INDEX_TABLE = f"{GOLD_SCHEMA}.gold_rag_retrieval_index"
AGENT_RUN_LOG_TABLE = f"{GOLD_SCHEMA}.gold_supplier_risk_agent_run_logs"

AGENT_EVAL_TABLE = f"{GOLD_SCHEMA}.gold_supplier_risk_agent_eval_results"
AGENT_MONITORING_TABLE = f"{GOLD_SCHEMA}.gold_supplier_risk_agent_monitoring_metrics"

MLFLOW_EXPERIMENT_NAME = "/Users/vigya.awasthi@tamu.edu/supplysage_ai_agent_monitoring"

PROJECT_NAME = "SupplySage AI"
AGENT_NAME = "supplysage_supplier_risk_agent"
AGENT_VERSION = "v1_manual_graph"
EVAL_QUESTION_SET_VERSION = "v1"
DEFAULT_TOP_K_FINAL = 8

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print("Notebook 28 config restored.")
print("MLflow experiment:", MLFLOW_EXPERIMENT_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())

required_tables = [
    CHAT_CONTEXT_TABLE,
    EMBEDDINGS_TABLE,
    RETRIEVAL_INDEX_TABLE,
    AGENT_RUN_LOG_TABLE
]

for table_name in required_tables:
    row_count = spark.table(table_name).count()
    print(f"{table_name}: {row_count} rows")

print("Config repair complete.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 28 — Replacement Cell 2
# Recreate lightweight supplier risk agent runtime inside Notebook 28
# ─────────────────────────────────────────────────────────────

import json
import re
import time
import traceback
import subprocess
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from pyspark.sql import functions as F

# ─────────────────────────────────────────────────────────────
# Runtime config
# ─────────────────────────────────────────────────────────────

TOP_K_CANDIDATES = 2000
TOP_K_FINAL = 8
ST_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384

# Confirm upstream config exists from Cell 1
required_config_vars = [
    "CHAT_CONTEXT_TABLE",
    "EMBEDDINGS_TABLE",
    "RETRIEVAL_INDEX_TABLE",
    "AGENT_RUN_LOG_TABLE",
    "AGENT_NAME",
    "AGENT_VERSION",
    "PROJECT_NAME"
]

missing_config = [v for v in required_config_vars if v not in globals()]
assert not missing_config, f"Missing config variables from Cell 1: {missing_config}"

print("Runtime config available.")

# ─────────────────────────────────────────────────────────────
# Load embedding model
# ─────────────────────────────────────────────────────────────

subprocess.run(
    ["pip", "install", "sentence-transformers", "--quiet"],
    check=True
)

from sentence_transformers import SentenceTransformer

print(f"Loading retrieval model: {ST_MODEL_NAME}")
retrieval_model = SentenceTransformer(ST_MODEL_NAME)

actual_dim = int(
    retrieval_model.get_embedding_dimension()
    if hasattr(retrieval_model, "get_embedding_dimension")
    else retrieval_model.get_sentence_embedding_dimension()
)

assert actual_dim == EMBEDDING_DIM, f"Expected embedding dim {EMBEDDING_DIM}, got {actual_dim}"
print(f"Retrieval model ready. Embedding dim: {actual_dim}")

# ─────────────────────────────────────────────────────────────
# Supplier resolver
# ─────────────────────────────────────────────────────────────

supplier_lookup_pd = (
    spark.table(CHAT_CONTEXT_TABLE)
    .select("supplier_id", "supplier_name")
    .dropDuplicates(["supplier_id"])
    .orderBy("supplier_id")
    .toPandas()
)

supplier_lookup_pd["supplier_id_upper"] = supplier_lookup_pd["supplier_id"].astype(str).str.upper()
supplier_lookup_pd["supplier_name_lower"] = supplier_lookup_pd["supplier_name"].astype(str).str.lower()

print(f"Loaded supplier resolver rows: {len(supplier_lookup_pd)}")

def resolve_supplier_from_question(question: str) -> dict:
    if question is None or len(str(question).strip()) == 0:
        return {
            "supplier_id": None,
            "supplier_name": None,
            "confidence": 0.0,
            "resolution_method": "empty_question",
            "message": "Question is empty."
        }

    question_text = str(question).strip()
    question_lower = question_text.lower()

    # Explicit supplier ID
    supplier_id_match = re.search(r"\bSUP[_\- ]?(\d+)\b", question_text, flags=re.IGNORECASE)

    if supplier_id_match:
        supplier_num = supplier_id_match.group(1)
        normalized_supplier_id = f"SUP_{supplier_num}"

        match_pd = supplier_lookup_pd[
            supplier_lookup_pd["supplier_id_upper"] == normalized_supplier_id.upper()
        ]

        if len(match_pd) > 0:
            row = match_pd.iloc[0]
            return {
                "supplier_id": row["supplier_id"],
                "supplier_name": row["supplier_name"],
                "confidence": 1.0,
                "resolution_method": "explicit_supplier_id",
                "message": f"Resolved explicit supplier ID {row['supplier_id']}."
            }

    # Exact supplier name substring
    for _, row in supplier_lookup_pd.iterrows():
        if row["supplier_name_lower"] in question_lower:
            return {
                "supplier_id": row["supplier_id"],
                "supplier_name": row["supplier_name"],
                "confidence": 0.95,
                "resolution_method": "supplier_name_exact_substring",
                "message": f"Resolved supplier name {row['supplier_name']}."
            }

    # First token match, useful for names like FlexPack
    for _, row in supplier_lookup_pd.iterrows():
        first_token = str(row["supplier_name"]).split()[0].lower()

        if len(first_token) >= 4 and first_token in question_lower:
            return {
                "supplier_id": row["supplier_id"],
                "supplier_name": row["supplier_name"],
                "confidence": 0.85,
                "resolution_method": "supplier_name_first_token",
                "message": f"Resolved supplier by first token {first_token}."
            }

    return {
        "supplier_id": None,
        "supplier_name": None,
        "confidence": 0.0,
        "resolution_method": "unresolved",
        "message": "Could not resolve supplier from question."
    }

# ─────────────────────────────────────────────────────────────
# Snapshot tool
# ─────────────────────────────────────────────────────────────

def parse_json_maybe(value):
    if value is None:
        return None

    if isinstance(value, dict) or isinstance(value, list):
        return value

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    try:
        return json.loads(value)
    except Exception:
        return None


def get_supplier_snapshot(supplier_id: str) -> dict:
    if supplier_id is None:
        return {
            "ok": False,
            "error": "supplier_id is required.",
            "supplier_id": supplier_id,
            "snapshot": None
        }

    supplier_id = str(supplier_id).strip().upper()

    rows = (
        spark.table(CHAT_CONTEXT_TABLE)
        .filter(F.col("supplier_id") == supplier_id)
        .limit(1)
        .collect()
    )

    if not rows:
        return {
            "ok": False,
            "error": f"No snapshot found for supplier_id={supplier_id}.",
            "supplier_id": supplier_id,
            "snapshot": None
        }

    row = rows[0].asDict(recursive=True)
    chat_context_json = parse_json_maybe(row.get("chat_context_json")) or {}

    snapshot = {
        "supplier_id": row.get("supplier_id"),
        "supplier_name": row.get("supplier_name"),
        "snapshot_date": row.get("snapshot_date"),
        "snapshot_timestamp": row.get("snapshot_timestamp"),
        "supplier_risk_score": row.get("supplier_risk_score"),
        "risk_band": row.get("risk_band"),
        "score_delta": row.get("score_delta"),
        "criticality_tier": row.get("criticality_tier"),
        "annual_spend": row.get("annual_spend"),
        "mapped_sku_count": row.get("mapped_sku_count"),
        "open_alert_count": row.get("open_alert_count"),
        "critical_or_high_alert_count": row.get("critical_or_high_alert_count"),
        "active_external_event_count": row.get("active_external_event_count"),
        "high_severity_event_count": row.get("high_severity_event_count"),
        "top_affected_sku_count": row.get("top_affected_sku_count"),
        "high_risk_sku_count": row.get("high_risk_sku_count"),
        "latest_external_event_date": row.get("latest_external_event_date"),
        "top_risk_driver": row.get("top_risk_driver"),
        "recommended_action": row.get("recommended_action"),
        "chat_context_text": row.get("chat_context_text"),
        "chat_context_json": chat_context_json
    }

    return {
        "ok": True,
        "error": None,
        "supplier_id": supplier_id,
        "supplier_name": snapshot.get("supplier_name"),
        "snapshot": snapshot
    }

# ─────────────────────────────────────────────────────────────
# Evidence retrieval + cleaning
# ─────────────────────────────────────────────────────────────

def clean_evidence_text(text, max_chars: int = 700) -> str:
    if text is None:
        return ""

    text = str(text)

    if '"raw_text"' in text or "'raw_text'" in text:
        raw_text_pos = text.find('"raw_text"')
        if raw_text_pos == -1:
            raw_text_pos = text.find("'raw_text'")

        prefix = text[:raw_text_pos].strip()

        if len(prefix) > 30:
            text = prefix + " [Raw payload omitted for readability.]"
        else:
            text = "Raw sanctions/compliance payload detected. Full payload omitted for readability."

    text = re.sub(
        r"<\?xml.*",
        "[Raw XML payload omitted for readability.]",
        text,
        flags=re.DOTALL
    )

    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    text = text.replace("| { [Raw payload omitted for readability.]", "| Raw payload omitted for readability.")
    text = text.replace("{ [Raw payload omitted for readability.]", "Raw payload omitted for readability.")

    if len(text) > max_chars:
        text = text[:max_chars].rstrip() + "..."

    return text


def retrieve_supplier_evidence(
    supplier_id: str,
    question: str,
    top_k_candidates: int = TOP_K_CANDIDATES,
    top_k_final: int = TOP_K_FINAL
) -> dict:
    if supplier_id is None or len(str(supplier_id).strip()) == 0:
        return {
            "ok": False,
            "error": "supplier_id is required.",
            "supplier_id": supplier_id,
            "evidence": []
        }

    if question is None or len(str(question).strip()) == 0:
        return {
            "ok": False,
            "error": "question is required.",
            "supplier_id": supplier_id,
            "evidence": []
        }

    supplier_id = str(supplier_id).strip().upper()
    question = str(question).strip()

    query_embedding = retrieval_model.encode(
        question,
        normalize_embeddings=True
    ).tolist()

    query_vector = np.array(query_embedding, dtype=np.float32)

    candidate_pd = (
        spark.table(EMBEDDINGS_TABLE)
        .filter(F.col("supplier_id") == supplier_id)
        .select(
            "chunk_id",
            "original_chunk_id",
            "chunk_type",
            "supplier_id",
            "sku_id",
            "source_name",
            "risk_category",
            "event_date",
            "severity",
            "freshness_weight",
            "source_url",
            "evidence_doc_id",
            "chunk_text",
            "embedding"
        )
        .orderBy(
            F.desc("freshness_weight"),
            F.desc("event_date")
        )
        .limit(top_k_candidates)
        .toPandas()
    )

    if len(candidate_pd) == 0:
        return {
            "ok": False,
            "error": f"No evidence found for supplier {supplier_id}.",
            "supplier_id": supplier_id,
            "candidate_count": 0,
            "evidence": []
        }

    embedding_matrix = np.vstack(
        candidate_pd["embedding"]
        .apply(lambda x: np.array(x, dtype=np.float32))
        .values
    )

    if embedding_matrix.shape[1] != len(query_vector):
        return {
            "ok": False,
            "error": f"Embedding dimension mismatch. Evidence dim={embedding_matrix.shape[1]}, query dim={len(query_vector)}",
            "supplier_id": supplier_id,
            "candidate_count": len(candidate_pd),
            "evidence": []
        }

    candidate_pd["cosine_similarity"] = embedding_matrix @ query_vector

    candidate_pd["freshness_weight"] = pd.to_numeric(
        candidate_pd["freshness_weight"],
        errors="coerce"
    ).fillna(0.5)

    candidate_pd["event_date_parsed"] = pd.to_datetime(
        candidate_pd["event_date"],
        errors="coerce"
    )

    today = pd.Timestamp.today().normalize()

    candidate_pd["days_since_event"] = (
        today - candidate_pd["event_date_parsed"]
    ).dt.days

    candidate_pd["recency_score"] = np.select(
        [
            candidate_pd["event_date_parsed"].isna(),
            candidate_pd["days_since_event"] <= 30,
            candidate_pd["days_since_event"] <= 90,
            candidate_pd["days_since_event"] <= 365,
        ],
        [
            0.35,
            1.00,
            0.85,
            0.65,
        ],
        default=0.25
    )

    severity_map = {
        "critical": 1.00,
        "high": 0.90,
        "medium": 0.55,
        "low": 0.30
    }

    candidate_pd["severity_score"] = (
        candidate_pd["severity"]
        .fillna("unknown")
        .astype(str)
        .str.lower()
        .map(severity_map)
        .fillna(0.40)
    )

    candidate_pd["chunk_type_score"] = np.select(
        [
            candidate_pd["chunk_type"].astype(str).str.lower().eq("risk_explanation"),
            candidate_pd["chunk_type"].astype(str).str.lower().eq("external_event"),
        ],
        [
            1.00,
            0.85,
        ],
        default=0.50
    )

    driver_sources = ["internal_risk_engine", "ofac", "cisa", "nws", "eia", "sec", "cpsc"]

    candidate_pd["driver_source_score"] = (
        candidate_pd["source_name"]
        .fillna("")
        .astype(str)
        .str.lower()
        .isin(driver_sources)
        .astype(float)
    )

    driver_categories = [
        "supplier_risk",
        "sanctions_compliance",
        "cyber",
        "logistics",
        "operational",
        "quality_recall",
        "weather",
        "energy"
    ]

    candidate_pd["driver_category_score"] = (
        candidate_pd["risk_category"]
        .fillna("")
        .astype(str)
        .str.lower()
        .isin(driver_categories)
        .astype(float)
    )

    candidate_pd["business_retrieval_score"] = (
        0.45 * candidate_pd["cosine_similarity"]
        + 0.15 * candidate_pd["freshness_weight"]
        + 0.15 * candidate_pd["severity_score"]
        + 0.10 * candidate_pd["recency_score"]
        + 0.10 * candidate_pd["driver_source_score"]
        + 0.05 * candidate_pd["driver_category_score"]
    )

    ranked_pd = (
        candidate_pd
        .sort_values(
            by=[
                "business_retrieval_score",
                "driver_source_score",
                "severity_score",
                "recency_score",
                "cosine_similarity"
            ],
            ascending=False
        )
        .head(top_k_final)
        .copy()
        .reset_index(drop=True)
    )

    evidence = []

    for idx, row in ranked_pd.iterrows():
        chunk_text = row.get("chunk_text")

        evidence.append({
            "rank": int(idx + 1),
            "chunk_id": row.get("chunk_id"),
            "original_chunk_id": row.get("original_chunk_id"),
            "chunk_type": row.get("chunk_type"),
            "supplier_id": row.get("supplier_id"),
            "sku_id": row.get("sku_id"),
            "source_name": row.get("source_name"),
            "risk_category": row.get("risk_category"),
            "event_date": str(row.get("event_date")) if pd.notna(row.get("event_date")) else None,
            "severity": row.get("severity"),
            "freshness_weight": float(row.get("freshness_weight")) if pd.notna(row.get("freshness_weight")) else None,
            "cosine_similarity": float(row.get("cosine_similarity")) if pd.notna(row.get("cosine_similarity")) else None,
            "business_retrieval_score": float(row.get("business_retrieval_score")) if pd.notna(row.get("business_retrieval_score")) else None,
            "source_url": row.get("source_url"),
            "evidence_doc_id": row.get("evidence_doc_id"),
            "cleaned_evidence_text": clean_evidence_text(chunk_text)
        })

    return {
        "ok": True,
        "error": None,
        "supplier_id": supplier_id,
        "question": question,
        "candidate_count": int(len(candidate_pd)),
        "returned_count": int(len(evidence)),
        "evidence": evidence
    }

# ─────────────────────────────────────────────────────────────
# Answer generation
# ─────────────────────────────────────────────────────────────

def generate_supplier_risk_answer(
    question: str,
    snapshot_result: dict,
    cleaned_evidence_result: dict
) -> dict:
    if not snapshot_result.get("ok"):
        return {
            "ok": False,
            "error": snapshot_result.get("error", "Supplier snapshot failed."),
            "answer": None
        }

    if not cleaned_evidence_result.get("ok"):
        return {
            "ok": False,
            "error": cleaned_evidence_result.get("error", "Evidence retrieval failed."),
            "answer": None
        }

    snapshot = snapshot_result["snapshot"]
    snapshot_json = snapshot.get("chat_context_json", {}) or {}
    evidence = cleaned_evidence_result.get("evidence", []) or []

    supplier_id = snapshot["supplier_id"]
    supplier_name = snapshot["supplier_name"]
    risk_score = snapshot["supplier_risk_score"]
    risk_band = snapshot["risk_band"]
    top_driver = snapshot["top_risk_driver"]
    recommended_action = snapshot["recommended_action"]

    top_skus = snapshot_json.get("top_affected_skus") or []
    active_events = snapshot_json.get("active_external_events") or []

    q_lower = str(question).lower()

    if "high risk" in q_lower and str(risk_band).lower() != "high":
        risk_opening = f"{supplier_name} is currently classified as {risk_band}, not high risk."
    else:
        risk_opening = f"{supplier_name} is currently classified as {risk_band}."

    event_lines = []

    for event in active_events[:5]:
        event_lines.append(
            f"- {event.get('source_name')} | {event.get('risk_category')} | "
            f"{event.get('severity')} | {event.get('event_date')}: "
            f"{event.get('event_title')}"
        )

    if not event_lines:
        event_lines = ["- No active external events found in the supplier snapshot."]

    sku_lines = []

    for sku in top_skus[:6]:
        sku_lines.append(
            f"- {sku.get('canonical_sku_id')}: "
            f"dependency={sku.get('dependency_percent')}, "
            f"stockout risk={sku.get('stockout_risk_score')}, "
            f"primary supplier={sku.get('is_primary_supplier')}, "
            f"alternate supplier={sku.get('alternate_supplier_id')}"
        )

    if not sku_lines:
        sku_lines = ["- No affected SKUs found in the supplier snapshot."]

    evidence_lines = []

    for ev in evidence[:6]:
        evidence_lines.append(
            f"[{ev.get('rank')}] {ev.get('source_name')} | "
            f"{ev.get('risk_category')} | {ev.get('severity')} | "
            f"{ev.get('event_date')} | {ev.get('cleaned_evidence_text')}"
        )

    if not evidence_lines:
        evidence_lines = ["No retrieved evidence available."]

    high_risk_sku_count = snapshot.get("high_risk_sku_count", 0)
    open_alert_count = snapshot.get("open_alert_count", 0)

    if high_risk_sku_count == 0 and open_alert_count == 0:
        operational_interpretation = (
            "This is currently an event-driven supplier monitoring case, "
            "not an immediate stockout or open-alert escalation. "
            "The affected SKUs do not currently show high stockout risk, "
            "so the right action is continued monitoring unless new external events or alerts appear."
        )
    else:
        operational_interpretation = (
            "This supplier needs closer operational review because there are either high-risk SKUs "
            "or open alerts associated with the supplier."
        )

    answer = f"""
Question:
{question.strip()}

Answer:
{risk_opening}

{supplier_name} ({supplier_id}) has a supplier risk score of {risk_score}. The current top risk driver is {top_driver}. The recommended action is: {recommended_action}

Why the supplier is being monitored:
- Active external events: {snapshot.get("active_external_event_count", 0)}
- High-severity external events: {snapshot.get("high_severity_event_count", 0)}
- Latest external event date: {snapshot.get("latest_external_event_date")}
- Affected SKUs: {snapshot.get("top_affected_sku_count", 0)}
- High-risk affected SKUs: {snapshot.get("high_risk_sku_count", 0)}
- Open alerts: {snapshot.get("open_alert_count", 0)}

Main external events:
{chr(10).join(event_lines)}

Top affected SKUs:
{chr(10).join(sku_lines)}

Supporting evidence:
{chr(10).join(evidence_lines)}

Recommendation:
{recommended_action}

Operational interpretation:
{operational_interpretation}
""".strip()

    return {
        "ok": True,
        "error": None,
        "supplier_id": supplier_id,
        "supplier_name": supplier_name,
        "answer": answer,
        "evidence_count": len(evidence),
        "risk_band": risk_band,
        "recommended_action": recommended_action
    }

# ─────────────────────────────────────────────────────────────
# Manual graph runner
# ─────────────────────────────────────────────────────────────

def run_supplier_risk_graph(question: str) -> dict:
    run_started_at = datetime.now(timezone.utc)

    state = {
        "question": question,
        "ok": None,
        "error": None,
        "trace": [],
        "run_started_at": run_started_at.isoformat(),
        "run_finished_at": None,
        "duration_seconds": None
    }

    try:
        # Node 1: resolve supplier
        resolver_result = resolve_supplier_from_question(question)

        state["resolver"] = resolver_result
        state["supplier_id"] = resolver_result.get("supplier_id")
        state["supplier_name"] = resolver_result.get("supplier_name")

        state["trace"].append({
            "node": "resolve_supplier",
            "ok": resolver_result.get("supplier_id") is not None,
            "method": resolver_result.get("resolution_method"),
            "confidence": resolver_result.get("confidence"),
            "message": resolver_result.get("message")
        })

        if resolver_result.get("supplier_id") is None:
            state["ok"] = False
            state["error"] = "supplier_unresolved"
            state["final_answer"] = (
                "I could not identify a specific supplier from the question. "
                "Please include a supplier ID like SUP_134 or a supplier name like FlexPack Industries."
            )
            return state

        # Node 2: snapshot
        snapshot_result = get_supplier_snapshot(state["supplier_id"])

        state["snapshot_result"] = snapshot_result

        state["trace"].append({
            "node": "get_supplier_snapshot",
            "ok": snapshot_result.get("ok"),
            "error": snapshot_result.get("error")
        })

        if not snapshot_result.get("ok"):
            state["ok"] = False
            state["error"] = snapshot_result.get("error")
            state["final_answer"] = f"I found supplier {state['supplier_id']}, but could not load its snapshot."
            return state

        # Node 3: retrieve evidence
        evidence_result = retrieve_supplier_evidence(
            supplier_id=state["supplier_id"],
            question=question,
            top_k_candidates=TOP_K_CANDIDATES,
            top_k_final=TOP_K_FINAL
        )

        state["cleaned_evidence_result"] = evidence_result

        state["trace"].append({
            "node": "retrieve_and_clean_supplier_evidence",
            "ok": evidence_result.get("ok"),
            "error": evidence_result.get("error"),
            "candidate_count": evidence_result.get("candidate_count"),
            "returned_count": evidence_result.get("returned_count")
        })

        if not evidence_result.get("ok"):
            state["ok"] = False
            state["error"] = evidence_result.get("error")
            state["final_answer"] = f"I loaded the supplier snapshot for {state.get('supplier_name')}, but could not retrieve supporting evidence."
            return state

        # Node 4: answer generation
        answer_result = generate_supplier_risk_answer(
            question=question,
            snapshot_result=snapshot_result,
            cleaned_evidence_result=evidence_result
        )

        state["answer_result"] = answer_result

        state["trace"].append({
            "node": "generate_supplier_risk_answer",
            "ok": answer_result.get("ok"),
            "error": answer_result.get("error"),
            "evidence_count": answer_result.get("evidence_count")
        })

        if answer_result.get("ok"):
            state["ok"] = True
            state["error"] = None
            state["final_answer"] = answer_result.get("answer")
            state["risk_band"] = answer_result.get("risk_band")
            state["recommended_action"] = answer_result.get("recommended_action")
            state["evidence_count"] = answer_result.get("evidence_count")
        else:
            state["ok"] = False
            state["error"] = answer_result.get("error")
            state["final_answer"] = "I could not generate a supplier risk answer."

    except Exception as e:
        state["ok"] = False
        state["error"] = str(e)
        state["final_answer"] = "The supplier risk graph encountered an unexpected error."
        state["exception_traceback"] = traceback.format_exc()

    finally:
        run_finished_at = datetime.now(timezone.utc)
        state["run_finished_at"] = run_finished_at.isoformat()
        state["duration_seconds"] = (run_finished_at - run_started_at).total_seconds()

    return state

# ─────────────────────────────────────────────────────────────
# Sanity test
# ─────────────────────────────────────────────────────────────

runtime_test_question = """
Why is supplier SUP_134 high risk?
Explain the main external events, affected SKUs, and recommended action.
"""

runtime_test_result = run_supplier_risk_graph(runtime_test_question)

print("Runtime bootstrap test:")
print("ok:", runtime_test_result.get("ok"))
print("error:", runtime_test_result.get("error"))
print("supplier_id:", runtime_test_result.get("supplier_id"))
print("supplier_name:", runtime_test_result.get("supplier_name"))
print("duration_seconds:", runtime_test_result.get("duration_seconds"))

print("\nTrace:")
for step in runtime_test_result.get("trace", []):
    print(step)

print("\nRuntime objects now available:")
print("run_supplier_risk_graph:", "run_supplier_risk_graph" in globals())
print("retrieval_model:", "retrieval_model" in globals())

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 28 — Cell 3
# MLflow-monitored supplier risk agent wrapper
# ─────────────────────────────────────────────────────────────

import json
import time
import mlflow
from datetime import datetime, timezone

assert "run_supplier_risk_graph" in globals(), "run_supplier_risk_graph missing. Rerun Replacement Cell 2."
assert "retrieval_model" in globals(), "retrieval_model missing. Rerun Replacement Cell 2."

def extract_agent_monitoring_metrics(agent_state: dict) -> dict:
    """
    Extract structured monitoring metrics from one agent run.
    These metrics are logged to MLflow and later saved to Delta.
    """

    trace = agent_state.get("trace") or []
    evidence = agent_state.get("cleaned_evidence_result", {}).get("evidence", []) or []

    source_names = []
    risk_categories = []
    severities = []

    for ev in evidence:
        if ev.get("source_name"):
            source_names.append(str(ev.get("source_name")))
        if ev.get("risk_category"):
            risk_categories.append(str(ev.get("risk_category")))
        if ev.get("severity"):
            severities.append(str(ev.get("severity")))

    final_answer = agent_state.get("final_answer") or ""

    metrics = {
        "agent_ok": int(bool(agent_state.get("ok"))),
        "duration_seconds": float(agent_state.get("duration_seconds") or 0.0),
        "trace_step_count": int(len(trace)),
        "evidence_count": int(agent_state.get("evidence_count") or len(evidence)),
        "supplier_resolved": int(agent_state.get("supplier_id") is not None),
        "has_final_answer": int(len(final_answer) > 0),
        "answer_char_length": int(len(final_answer)),
        "unique_source_count": int(len(set(source_names))),
        "unique_risk_category_count": int(len(set(risk_categories))),
        "high_severity_evidence_count": int(sum(1 for s in severities if s.lower() == "high")),
        "medium_severity_evidence_count": int(sum(1 for s in severities if s.lower() == "medium")),
        "retrieval_candidate_count": int(
            agent_state.get("cleaned_evidence_result", {}).get("candidate_count") or 0
        ),
        "retrieval_returned_count": int(
            agent_state.get("cleaned_evidence_result", {}).get("returned_count") or 0
        )
    }

    return metrics


# Optional MLflow trace decorator.
# If the MLflow trace decorator behaves differently in this Databricks runtime,
# the start_run logging below still works.
if hasattr(mlflow, "trace"):
    @mlflow.trace(name="supplysage_supplier_risk_graph", span_type="CHAIN")
    def traced_supplier_risk_graph(question: str) -> dict:
        return run_supplier_risk_graph(question)
else:
    def traced_supplier_risk_graph(question: str) -> dict:
        return run_supplier_risk_graph(question)


def run_supplier_risk_agent_with_mlflow(question: str) -> dict:
    """
    Runs the supplier risk graph and logs to MLflow:
    - params
    - numeric monitoring metrics
    - trace JSON artifact
    - resolver artifact
    - evidence artifact
    - final answer artifact
    """

    wrapper_started_at = datetime.now(timezone.utc)
    timer_start = time.time()

    with mlflow.start_run(run_name=f"{AGENT_NAME}_{AGENT_VERSION}") as run:
        mlflow.set_tag("project_name", PROJECT_NAME)
        mlflow.set_tag("agent_name", AGENT_NAME)
        mlflow.set_tag("agent_version", AGENT_VERSION)
        mlflow.set_tag("run_type", "agent_monitoring")
        mlflow.set_tag("notebook", "28_mlflow_agent_monitoring_evaluation")
        mlflow.set_tag("eval_question_set_version", EVAL_QUESTION_SET_VERSION)

        mlflow.log_param("question", question)
        mlflow.log_param("agent_version", AGENT_VERSION)
        mlflow.log_param("top_k_final", DEFAULT_TOP_K_FINAL)

        # Run traced graph
        agent_state = traced_supplier_risk_graph(question)

        wrapper_runtime_seconds = time.time() - timer_start

        # Attach MLflow metadata to state
        agent_state["mlflow_run_id"] = run.info.run_id
        agent_state["mlflow_experiment_id"] = run.info.experiment_id
        agent_state["monitoring_logged_at"] = datetime.now(timezone.utc).isoformat()
        agent_state["wrapper_runtime_seconds"] = float(wrapper_runtime_seconds)

        metrics = extract_agent_monitoring_metrics(agent_state)
        metrics["wrapper_runtime_seconds"] = float(wrapper_runtime_seconds)

        # Log numeric metrics
        for metric_name, metric_value in metrics.items():
            mlflow.log_metric(metric_name, metric_value)

        # Log post-run params
        mlflow.log_param("supplier_id", agent_state.get("supplier_id"))
        mlflow.log_param("supplier_name", agent_state.get("supplier_name"))
        mlflow.log_param("risk_band", agent_state.get("risk_band"))
        mlflow.log_param("recommended_action", agent_state.get("recommended_action"))
        mlflow.log_param("ok", agent_state.get("ok"))
        mlflow.log_param("error", agent_state.get("error"))

        # Log artifacts
        mlflow.log_dict(agent_state.get("trace") or [], "trace/agent_trace.json")
        mlflow.log_dict(agent_state.get("resolver") or {}, "trace/resolver.json")
        mlflow.log_dict(agent_state.get("cleaned_evidence_result") or {}, "trace/cleaned_evidence.json")
        mlflow.log_text(agent_state.get("final_answer") or "", "outputs/final_answer.txt")

        monitoring_payload = {
            "wrapper_started_at": wrapper_started_at.isoformat(),
            "wrapper_finished_at": datetime.now(timezone.utc).isoformat(),
            "mlflow_run_id": run.info.run_id,
            "mlflow_experiment_id": run.info.experiment_id,
            "question": question,
            "supplier_id": agent_state.get("supplier_id"),
            "supplier_name": agent_state.get("supplier_name"),
            "ok": agent_state.get("ok"),
            "error": agent_state.get("error"),
            "risk_band": agent_state.get("risk_band"),
            "recommended_action": agent_state.get("recommended_action"),
            "metrics": metrics
        }

        mlflow.log_dict(monitoring_payload, "monitoring/monitoring_payload.json")

    return agent_state


# Test MLflow-monitored agent run
mlflow_test_question = """
Why is supplier SUP_134 high risk?
Explain the main external events, affected SKUs, and recommended action.
"""

mlflow_agent_result = run_supplier_risk_agent_with_mlflow(mlflow_test_question)

print("MLflow-monitored agent result:")
print("ok:", mlflow_agent_result.get("ok"))
print("error:", mlflow_agent_result.get("error"))
print("supplier_id:", mlflow_agent_result.get("supplier_id"))
print("supplier_name:", mlflow_agent_result.get("supplier_name"))
print("risk_band:", mlflow_agent_result.get("risk_band"))
print("evidence_count:", mlflow_agent_result.get("evidence_count"))
print("duration_seconds:", mlflow_agent_result.get("duration_seconds"))
print("wrapper_runtime_seconds:", mlflow_agent_result.get("wrapper_runtime_seconds"))
print("mlflow_run_id:", mlflow_agent_result.get("mlflow_run_id"))

print("\nTrace:")
for step in mlflow_agent_result.get("trace", []):
    print(step)

print("\n" + "=" * 100)
print(mlflow_agent_result.get("final_answer"))

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 28 — Cell 3A
# Patch runtime fallback for missing top_risk_driver / recommended_action
# ─────────────────────────────────────────────────────────────

import re
import json
import pandas as pd
from pyspark.sql import functions as F

def extract_line_value(text: str, label: str):
    """
    Extracts values from lines like:
    Top risk driver: External events
    Recommended action: Monitor — no immediate action required
    """

    if text is None:
        return None

    text = str(text)

    pattern = rf"{re.escape(label)}\s*:\s*(.+)"
    match = re.search(pattern, text, flags=re.IGNORECASE)

    if not match:
        return None

    value = match.group(1).strip()

    # Stop if multiple fields accidentally got captured on same line
    value = re.split(
        r"\s+\|\s+|Open alerts:|Active external events:|Top affected SKU count:|Risk explanation:",
        value
    )[0].strip()

    return value if value else None


def extract_from_internal_evidence(evidence_result: dict, field_name: str):
    """
    Fallback from internal_risk_engine evidence text:
    Top driver: External events
    Recommended action: Monitor — no immediate action required
    """

    evidence = evidence_result.get("evidence", []) if evidence_result else []

    for ev in evidence:
        text = ev.get("cleaned_evidence_text") or ""

        if str(ev.get("source_name")).lower() == "internal_risk_engine":
            if field_name == "top_risk_driver":
                match = re.search(
                    r"Top driver:\s*([^|]+)",
                    text,
                    flags=re.IGNORECASE
                )
                if match:
                    return match.group(1).strip()

            if field_name == "recommended_action":
                match = re.search(
                    r"Recommended action:\s*(.+)$",
                    text,
                    flags=re.IGNORECASE
                )
                if match:
                    return match.group(1).strip()

    return None


# Override get_supplier_snapshot with robust fallback logic
def get_supplier_snapshot(supplier_id: str) -> dict:
    if supplier_id is None:
        return {
            "ok": False,
            "error": "supplier_id is required.",
            "supplier_id": supplier_id,
            "snapshot": None
        }

    supplier_id = str(supplier_id).strip().upper()

    rows = (
        spark.table(CHAT_CONTEXT_TABLE)
        .filter(F.col("supplier_id") == supplier_id)
        .limit(1)
        .collect()
    )

    if not rows:
        return {
            "ok": False,
            "error": f"No snapshot found for supplier_id={supplier_id}.",
            "supplier_id": supplier_id,
            "snapshot": None
        }

    row = rows[0].asDict(recursive=True)

    chat_context_text = row.get("chat_context_text")
    chat_context_json = parse_json_maybe(row.get("chat_context_json")) or {}

    # Robust fallbacks
    top_risk_driver = (
        row.get("top_risk_driver")
        or chat_context_json.get("top_risk_driver")
        or chat_context_json.get("risk_driver")
        or extract_line_value(chat_context_text, "Top risk driver")
    )

    recommended_action = (
        row.get("recommended_action")
        or chat_context_json.get("recommended_action")
        or chat_context_json.get("action")
        or extract_line_value(chat_context_text, "Recommended action")
    )

    snapshot = {
        "supplier_id": row.get("supplier_id"),
        "supplier_name": row.get("supplier_name"),
        "snapshot_date": row.get("snapshot_date"),
        "snapshot_timestamp": row.get("snapshot_timestamp"),
        "supplier_risk_score": row.get("supplier_risk_score"),
        "risk_band": row.get("risk_band"),
        "score_delta": row.get("score_delta"),
        "criticality_tier": row.get("criticality_tier"),
        "annual_spend": row.get("annual_spend"),
        "mapped_sku_count": row.get("mapped_sku_count"),
        "open_alert_count": row.get("open_alert_count"),
        "critical_or_high_alert_count": row.get("critical_or_high_alert_count"),
        "active_external_event_count": row.get("active_external_event_count"),
        "high_severity_event_count": row.get("high_severity_event_count"),
        "top_affected_sku_count": row.get("top_affected_sku_count"),
        "high_risk_sku_count": row.get("high_risk_sku_count"),
        "latest_external_event_date": row.get("latest_external_event_date"),
        "top_risk_driver": top_risk_driver,
        "recommended_action": recommended_action,
        "chat_context_text": chat_context_text,
        "chat_context_json": chat_context_json
    }

    return {
        "ok": True,
        "error": None,
        "supplier_id": supplier_id,
        "supplier_name": snapshot.get("supplier_name"),
        "snapshot": snapshot
    }


# Override answer generator with final fallback from evidence
def generate_supplier_risk_answer(
    question: str,
    snapshot_result: dict,
    cleaned_evidence_result: dict
) -> dict:
    if not snapshot_result.get("ok"):
        return {
            "ok": False,
            "error": snapshot_result.get("error", "Supplier snapshot failed."),
            "answer": None
        }

    if not cleaned_evidence_result.get("ok"):
        return {
            "ok": False,
            "error": cleaned_evidence_result.get("error", "Evidence retrieval failed."),
            "answer": None
        }

    snapshot = snapshot_result["snapshot"]
    snapshot_json = snapshot.get("chat_context_json", {}) or {}
    evidence = cleaned_evidence_result.get("evidence", []) or []

    supplier_id = snapshot["supplier_id"]
    supplier_name = snapshot["supplier_name"]
    risk_score = snapshot["supplier_risk_score"]
    risk_band = snapshot["risk_band"]

    top_driver = (
        snapshot.get("top_risk_driver")
        or extract_from_internal_evidence(cleaned_evidence_result, "top_risk_driver")
        or "Unknown"
    )

    recommended_action = (
        snapshot.get("recommended_action")
        or extract_from_internal_evidence(cleaned_evidence_result, "recommended_action")
        or "Review supplier risk evidence and monitor for changes."
    )

    top_skus = snapshot_json.get("top_affected_skus") or []
    active_events = snapshot_json.get("active_external_events") or []

    q_lower = str(question).lower()

    if "high risk" in q_lower and str(risk_band).lower() != "high":
        risk_opening = f"{supplier_name} is currently classified as {risk_band}, not high risk."
    else:
        risk_opening = f"{supplier_name} is currently classified as {risk_band}."

    event_lines = []

    for event in active_events[:5]:
        event_lines.append(
            f"- {event.get('source_name')} | {event.get('risk_category')} | "
            f"{event.get('severity')} | {event.get('event_date')}: "
            f"{event.get('event_title')}"
        )

    if not event_lines:
        event_lines = ["- No active external events found in the supplier snapshot."]

    sku_lines = []

    for sku in top_skus[:6]:
        sku_lines.append(
            f"- {sku.get('canonical_sku_id')}: "
            f"dependency={sku.get('dependency_percent')}, "
            f"stockout risk={sku.get('stockout_risk_score')}, "
            f"primary supplier={sku.get('is_primary_supplier')}, "
            f"alternate supplier={sku.get('alternate_supplier_id')}"
        )

    if not sku_lines:
        sku_lines = ["- No affected SKUs found in the supplier snapshot."]

    evidence_lines = []

    for ev in evidence[:6]:
        evidence_lines.append(
            f"[{ev.get('rank')}] {ev.get('source_name')} | "
            f"{ev.get('risk_category')} | {ev.get('severity')} | "
            f"{ev.get('event_date')} | {ev.get('cleaned_evidence_text')}"
        )

    if not evidence_lines:
        evidence_lines = ["No retrieved evidence available."]

    high_risk_sku_count = snapshot.get("high_risk_sku_count", 0)
    open_alert_count = snapshot.get("open_alert_count", 0)

    if high_risk_sku_count == 0 and open_alert_count == 0:
        operational_interpretation = (
            "This is currently an event-driven supplier monitoring case, "
            "not an immediate stockout or open-alert escalation. "
            "The affected SKUs do not currently show high stockout risk, "
            "so the right action is continued monitoring unless new external events or alerts appear."
        )
    else:
        operational_interpretation = (
            "This supplier needs closer operational review because there are either high-risk SKUs "
            "or open alerts associated with the supplier."
        )

    answer = f"""
Question:
{question.strip()}

Answer:
{risk_opening}

{supplier_name} ({supplier_id}) has a supplier risk score of {risk_score}. The current top risk driver is {top_driver}. The recommended action is: {recommended_action}

Why the supplier is being monitored:
- Active external events: {snapshot.get("active_external_event_count", 0)}
- High-severity external events: {snapshot.get("high_severity_event_count", 0)}
- Latest external event date: {snapshot.get("latest_external_event_date")}
- Affected SKUs: {snapshot.get("top_affected_sku_count", 0)}
- High-risk affected SKUs: {snapshot.get("high_risk_sku_count", 0)}
- Open alerts: {snapshot.get("open_alert_count", 0)}

Main external events:
{chr(10).join(event_lines)}

Top affected SKUs:
{chr(10).join(sku_lines)}

Supporting evidence:
{chr(10).join(evidence_lines)}

Recommendation:
{recommended_action}

Operational interpretation:
{operational_interpretation}
""".strip()

    return {
        "ok": True,
        "error": None,
        "supplier_id": supplier_id,
        "supplier_name": supplier_name,
        "answer": answer,
        "evidence_count": len(evidence),
        "risk_band": risk_band,
        "recommended_action": recommended_action,
        "top_risk_driver": top_driver
    }


# Sanity test after patch
patched_test_question = """
Why is supplier SUP_134 high risk?
Explain the main external events, affected SKUs, and recommended action.
"""

patched_result = run_supplier_risk_graph(patched_test_question)

print("Patched runtime test:")
print("ok:", patched_result.get("ok"))
print("error:", patched_result.get("error"))
print("supplier_id:", patched_result.get("supplier_id"))
print("supplier_name:", patched_result.get("supplier_name"))
print("risk_band:", patched_result.get("risk_band"))
print("recommended_action:", patched_result.get("recommended_action"))

print("\nAnswer preview:")
print(patched_result.get("final_answer")[:1000])

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 28 — Cell 3B
# Rerun MLflow-monitored agent after patch
# ─────────────────────────────────────────────────────────────

patched_mlflow_test_question = """
Why is supplier SUP_134 high risk?
Explain the main external events, affected SKUs, and recommended action.
"""

patched_mlflow_agent_result = run_supplier_risk_agent_with_mlflow(
    patched_mlflow_test_question
)

print("Patched MLflow-monitored agent result:")
print("ok:", patched_mlflow_agent_result.get("ok"))
print("error:", patched_mlflow_agent_result.get("error"))
print("supplier_id:", patched_mlflow_agent_result.get("supplier_id"))
print("supplier_name:", patched_mlflow_agent_result.get("supplier_name"))
print("risk_band:", patched_mlflow_agent_result.get("risk_band"))
print("recommended_action:", patched_mlflow_agent_result.get("recommended_action"))
print("evidence_count:", patched_mlflow_agent_result.get("evidence_count"))
print("duration_seconds:", patched_mlflow_agent_result.get("duration_seconds"))
print("wrapper_runtime_seconds:", patched_mlflow_agent_result.get("wrapper_runtime_seconds"))
print("mlflow_run_id:", patched_mlflow_agent_result.get("mlflow_run_id"))

print("\nTrace:")
for step in patched_mlflow_agent_result.get("trace", []):
    print(step)

print("\n" + "=" * 100)
print(patched_mlflow_agent_result.get("final_answer")[:1800])

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 28 — Cell 4
# Save MLflow-monitored agent metrics to Delta
# ─────────────────────────────────────────────────────────────

import json
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    BooleanType,
    TimestampType
)

assert "patched_mlflow_agent_result" in globals(), "patched_mlflow_agent_result missing. Rerun Cell 3B."
assert "extract_agent_monitoring_metrics" in globals(), "extract_agent_monitoring_metrics missing. Rerun Cell 3."

AGENT_MONITORING_TABLE = "supplysage_gold.gold_supplier_risk_agent_monitoring_metrics"

def safe_json_dumps(obj):
    return json.dumps(obj, default=str, ensure_ascii=False)

agent_state = patched_mlflow_agent_result
metrics = extract_agent_monitoring_metrics(agent_state)

evidence = agent_state.get("cleaned_evidence_result", {}).get("evidence", []) or []
trace = agent_state.get("trace") or []

source_names = [str(ev.get("source_name")) for ev in evidence if ev.get("source_name")]
risk_categories = [str(ev.get("risk_category")) for ev in evidence if ev.get("risk_category")]
severities = [str(ev.get("severity")) for ev in evidence if ev.get("severity")]

final_answer = agent_state.get("final_answer") or ""

# Simple quality/monitoring flags
assumption_corrected_flag = int(
    "not high risk" in final_answer.lower()
    or "not classified as high" in final_answer.lower()
)

none_leakage_flag = int(
    "recommended action is: none" in final_answer.lower()
    or "recommendation:\nnone" in final_answer.lower()
    or "top risk driver is none" in final_answer.lower()
)

raw_payload_leakage_flag = int(
    "<?xml" in final_answer.lower()
    or "<sdnlist" in final_answer.lower()
    or '"raw_text"' in final_answer.lower()
)

monitoring_row = [{
    "monitoring_event_id": f"monitor_{agent_state.get('supplier_id')}_{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}",
    "mlflow_run_id": agent_state.get("mlflow_run_id"),
    "mlflow_experiment_id": agent_state.get("mlflow_experiment_id"),
    "project_name": PROJECT_NAME,
    "agent_name": AGENT_NAME,
    "agent_version": AGENT_VERSION,
    "question": agent_state.get("question"),
    "ok": bool(agent_state.get("ok")),
    "error": agent_state.get("error"),
    "supplier_id": agent_state.get("supplier_id"),
    "supplier_name": agent_state.get("supplier_name"),
    "risk_band": agent_state.get("risk_band"),
    "recommended_action": agent_state.get("recommended_action"),
    "duration_seconds": float(agent_state.get("duration_seconds") or 0.0),
    "wrapper_runtime_seconds": float(agent_state.get("wrapper_runtime_seconds") or 0.0),
    "trace_step_count": int(metrics.get("trace_step_count") or 0),
    "evidence_count": int(metrics.get("evidence_count") or 0),
    "retrieval_candidate_count": int(metrics.get("retrieval_candidate_count") or 0),
    "retrieval_returned_count": int(metrics.get("retrieval_returned_count") or 0),
    "unique_source_count": int(metrics.get("unique_source_count") or 0),
    "unique_risk_category_count": int(metrics.get("unique_risk_category_count") or 0),
    "high_severity_evidence_count": int(metrics.get("high_severity_evidence_count") or 0),
    "medium_severity_evidence_count": int(metrics.get("medium_severity_evidence_count") or 0),
    "answer_char_length": int(metrics.get("answer_char_length") or 0),
    "supplier_resolved": int(metrics.get("supplier_resolved") or 0),
    "has_final_answer": int(metrics.get("has_final_answer") or 0),
    "assumption_corrected_flag": assumption_corrected_flag,
    "none_leakage_flag": none_leakage_flag,
    "raw_payload_leakage_flag": raw_payload_leakage_flag,
    "source_names_json": safe_json_dumps(sorted(list(set(source_names)))),
    "risk_categories_json": safe_json_dumps(sorted(list(set(risk_categories)))),
    "severities_json": safe_json_dumps(sorted(list(set(severities)))),
    "trace_json": safe_json_dumps(trace),
    "evidence_json": safe_json_dumps(evidence),
    "final_answer": final_answer,
    "run_started_at": agent_state.get("run_started_at"),
    "run_finished_at": agent_state.get("run_finished_at"),
    "monitoring_logged_at": datetime.now(timezone.utc)
}]

monitoring_schema = StructType([
    StructField("monitoring_event_id", StringType(), False),
    StructField("mlflow_run_id", StringType(), True),
    StructField("mlflow_experiment_id", StringType(), True),
    StructField("project_name", StringType(), True),
    StructField("agent_name", StringType(), True),
    StructField("agent_version", StringType(), True),
    StructField("question", StringType(), True),
    StructField("ok", BooleanType(), True),
    StructField("error", StringType(), True),
    StructField("supplier_id", StringType(), True),
    StructField("supplier_name", StringType(), True),
    StructField("risk_band", StringType(), True),
    StructField("recommended_action", StringType(), True),
    StructField("duration_seconds", DoubleType(), True),
    StructField("wrapper_runtime_seconds", DoubleType(), True),
    StructField("trace_step_count", IntegerType(), True),
    StructField("evidence_count", IntegerType(), True),
    StructField("retrieval_candidate_count", IntegerType(), True),
    StructField("retrieval_returned_count", IntegerType(), True),
    StructField("unique_source_count", IntegerType(), True),
    StructField("unique_risk_category_count", IntegerType(), True),
    StructField("high_severity_evidence_count", IntegerType(), True),
    StructField("medium_severity_evidence_count", IntegerType(), True),
    StructField("answer_char_length", IntegerType(), True),
    StructField("supplier_resolved", IntegerType(), True),
    StructField("has_final_answer", IntegerType(), True),
    StructField("assumption_corrected_flag", IntegerType(), True),
    StructField("none_leakage_flag", IntegerType(), True),
    StructField("raw_payload_leakage_flag", IntegerType(), True),
    StructField("source_names_json", StringType(), True),
    StructField("risk_categories_json", StringType(), True),
    StructField("severities_json", StringType(), True),
    StructField("trace_json", StringType(), True),
    StructField("evidence_json", StringType(), True),
    StructField("final_answer", StringType(), True),
    StructField("run_started_at", StringType(), True),
    StructField("run_finished_at", StringType(), True),
    StructField("monitoring_logged_at", TimestampType(), True),
])

monitoring_df = spark.createDataFrame(monitoring_row, schema=monitoring_schema)

(
    monitoring_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit("28_mlflow_agent_monitoring_evaluation"))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(AGENT_MONITORING_TABLE)
)

print(f"Saved monitoring metrics to: {AGENT_MONITORING_TABLE}")

display(
    spark.table(AGENT_MONITORING_TABLE)
    .select(
        "monitoring_event_id",
        "mlflow_run_id",
        "supplier_id",
        "supplier_name",
        "ok",
        "risk_band",
        "recommended_action",
        "duration_seconds",
        "wrapper_runtime_seconds",
        "trace_step_count",
        "evidence_count",
        "unique_source_count",
        "assumption_corrected_flag",
        "none_leakage_flag",
        "raw_payload_leakage_flag",
        "monitoring_logged_at"
    )
    .orderBy(F.desc("monitoring_logged_at"))
    .limit(10)
)

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 28 — Replacement Cell 5
# Build supplier risk agent evaluation question set
# Fixed for actual snapshot columns:
# - final_top_risk_driver
# - final_recommended_action
# ─────────────────────────────────────────────────────────────

from pyspark.sql import functions as F
import pandas as pd
import json

assert "CHAT_CONTEXT_TABLE" in globals(), "CHAT_CONTEXT_TABLE missing. Rerun Cell 2A."

snapshot_df = spark.table(CHAT_CONTEXT_TABLE)
available_cols = set(snapshot_df.columns)

print("Snapshot columns:")
print(snapshot_df.columns)

def col_or_null(col_name, cast_type="string"):
    if col_name in available_cols:
        return F.col(col_name).cast(cast_type)
    return F.lit(None).cast(cast_type)

def json_or_null(json_col, json_path):
    if json_col in available_cols:
        return F.get_json_object(F.col(json_col), json_path)
    return F.lit(None).cast("string")

# Build a robust eval source view with normalized field names
supplier_snapshot_pd = (
    snapshot_df
    .select(
        F.col("supplier_id"),
        F.col("supplier_name"),
        F.col("supplier_risk_score"),
        F.col("risk_band"),
        F.coalesce(
            col_or_null("final_top_risk_driver"),
            col_or_null("top_risk_driver"),
            json_or_null("chat_context_json", "$.top_risk_driver")
        ).alias("top_risk_driver"),
        F.coalesce(
            col_or_null("final_recommended_action"),
            col_or_null("recommended_action"),
            json_or_null("chat_context_json", "$.recommended_action")
        ).alias("recommended_action"),
        F.col("active_external_event_count"),
        F.col("top_affected_sku_count")
    )
    .dropDuplicates(["supplier_id"])
    .orderBy(F.desc("supplier_risk_score"))
    .toPandas()
)

assert len(supplier_snapshot_pd) > 0, "No supplier snapshots found."

# Fill nulls for safer eval question construction
supplier_snapshot_pd["top_risk_driver"] = supplier_snapshot_pd["top_risk_driver"].fillna("Unknown")
supplier_snapshot_pd["recommended_action"] = supplier_snapshot_pd["recommended_action"].fillna("Review supplier risk evidence and monitor for changes.")

# Pick representative suppliers
top_supplier = supplier_snapshot_pd.iloc[0]

medium_candidates = supplier_snapshot_pd[
    supplier_snapshot_pd["risk_band"].astype(str).str.lower() == "medium"
]

medium_supplier = (
    medium_candidates.iloc[0]
    if len(medium_candidates) > 0
    else supplier_snapshot_pd.iloc[0]
)

low_candidates = supplier_snapshot_pd[
    supplier_snapshot_pd["risk_band"].astype(str).str.lower().isin(["low", "minimal"])
]

low_supplier = (
    low_candidates.iloc[0]
    if len(low_candidates) > 0
    else supplier_snapshot_pd.iloc[-1]
)

# Always keep SUP_134 because it is our known monitored test case
known_supplier_id = "SUP_134"
known_supplier_rows = supplier_snapshot_pd[
    supplier_snapshot_pd["supplier_id"] == known_supplier_id
]

known_supplier = (
    known_supplier_rows.iloc[0]
    if len(known_supplier_rows) > 0
    else medium_supplier
)

eval_questions = [
    {
        "eval_id": "eval_001_assumption_correction",
        "question": f"Why is supplier {known_supplier['supplier_id']} high risk? Explain the main external events, affected SKUs, and recommended action.",
        "expected_supplier_id": known_supplier["supplier_id"],
        "expected_supplier_name": known_supplier["supplier_name"],
        "expected_risk_band": known_supplier["risk_band"],
        "expected_terms": [
            str(known_supplier["supplier_name"]),
            str(known_supplier["risk_band"]),
            "external",
            "SKU",
            "recommend"
        ],
        "expects_supplier_resolution": True,
        "expects_assumption_correction": str(known_supplier["risk_band"]).lower() != "high",
        "expects_evidence": True,
        "eval_category": "assumption_correction"
    },
    {
        "eval_id": "eval_002_supplier_name_lookup",
        "question": f"What is happening with {known_supplier['supplier_name']}?",
        "expected_supplier_id": known_supplier["supplier_id"],
        "expected_supplier_name": known_supplier["supplier_name"],
        "expected_risk_band": known_supplier["risk_band"],
        "expected_terms": [
            str(known_supplier["supplier_name"]),
            str(known_supplier["risk_band"]),
            "risk"
        ],
        "expects_supplier_resolution": True,
        "expects_assumption_correction": False,
        "expects_evidence": True,
        "eval_category": "supplier_name_lookup"
    },
    {
        "eval_id": "eval_003_top_supplier_summary",
        "question": f"Summarize the supplier risk for {top_supplier['supplier_id']}. What should operations do next?",
        "expected_supplier_id": top_supplier["supplier_id"],
        "expected_supplier_name": top_supplier["supplier_name"],
        "expected_risk_band": top_supplier["risk_band"],
        "expected_terms": [
            str(top_supplier["supplier_name"]),
            str(top_supplier["risk_band"]),
            "risk",
            "recommend"
        ],
        "expects_supplier_resolution": True,
        "expects_assumption_correction": False,
        "expects_evidence": True,
        "eval_category": "risk_summary"
    },
    {
        "eval_id": "eval_004_sku_dependency",
        "question": f"Which SKUs are most exposed to supplier {known_supplier['supplier_id']}?",
        "expected_supplier_id": known_supplier["supplier_id"],
        "expected_supplier_name": known_supplier["supplier_name"],
        "expected_risk_band": known_supplier["risk_band"],
        "expected_terms": [
            str(known_supplier["supplier_name"]),
            "SKU",
            "dependency"
        ],
        "expects_supplier_resolution": True,
        "expects_assumption_correction": False,
        "expects_evidence": True,
        "eval_category": "sku_dependency"
    },
    {
        "eval_id": "eval_005_external_events",
        "question": f"What external events are driving risk for {known_supplier['supplier_name']}?",
        "expected_supplier_id": known_supplier["supplier_id"],
        "expected_supplier_name": known_supplier["supplier_name"],
        "expected_risk_band": known_supplier["risk_band"],
        "expected_terms": [
            str(known_supplier["supplier_name"]),
            "external",
            "cisa",
            "ofac"
        ],
        "expects_supplier_resolution": True,
        "expects_assumption_correction": False,
        "expects_evidence": True,
        "eval_category": "external_events"
    },
    {
        "eval_id": "eval_006_low_supplier_check",
        "question": f"Does supplier {low_supplier['supplier_id']} need escalation?",
        "expected_supplier_id": low_supplier["supplier_id"],
        "expected_supplier_name": low_supplier["supplier_name"],
        "expected_risk_band": low_supplier["risk_band"],
        "expected_terms": [
            str(low_supplier["supplier_name"]),
            str(low_supplier["risk_band"]),
            "risk"
        ],
        "expects_supplier_resolution": True,
        "expects_assumption_correction": False,
        "expects_evidence": True,
        "eval_category": "escalation_check"
    },
    {
        "eval_id": "eval_007_partial_name",
        "question": f"Explain risk for {str(known_supplier['supplier_name']).split()[0]}",
        "expected_supplier_id": known_supplier["supplier_id"],
        "expected_supplier_name": known_supplier["supplier_name"],
        "expected_risk_band": known_supplier["risk_band"],
        "expected_terms": [
            str(known_supplier["supplier_name"]),
            str(known_supplier["risk_band"]),
            "risk"
        ],
        "expects_supplier_resolution": True,
        "expects_assumption_correction": False,
        "expects_evidence": True,
        "eval_category": "partial_name_resolution"
    },
    {
        "eval_id": "eval_008_unresolved_supplier",
        "question": "Which supplier should I monitor today?",
        "expected_supplier_id": None,
        "expected_supplier_name": None,
        "expected_risk_band": None,
        "expected_terms": [
            "specific supplier",
            "supplier ID"
        ],
        "expects_supplier_resolution": False,
        "expects_assumption_correction": False,
        "expects_evidence": False,
        "eval_category": "unresolved_supplier"
    }
]

agent_eval_questions = eval_questions
eval_questions_pd = pd.DataFrame(agent_eval_questions)

print("Evaluation question set created.")
print("Question count:", len(agent_eval_questions))

display(eval_questions_pd[
    [
        "eval_id",
        "eval_category",
        "question",
        "expected_supplier_id",
        "expected_supplier_name",
        "expected_risk_band",
        "expects_supplier_resolution",
        "expects_assumption_correction",
        "expects_evidence"
    ]
])

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 28 — Cell 6
# Run batch MLflow-monitored agent evaluation
# ─────────────────────────────────────────────────────────────

import pandas as pd
import json
import time
from datetime import datetime, timezone

assert "agent_eval_questions" in globals(), "agent_eval_questions missing. Rerun Replacement Cell 5."
assert "run_supplier_risk_agent_with_mlflow" in globals(), "run_supplier_risk_agent_with_mlflow missing. Rerun Cell 3."
assert "extract_agent_monitoring_metrics" in globals(), "extract_agent_monitoring_metrics missing. Rerun Cell 3."

def score_agent_response(eval_case: dict, agent_state: dict) -> dict:
    """
    Deterministic evaluation checks:
    - supplier resolution correctness
    - risk band mention
    - expected term coverage
    - evidence retrieval
    - assumption correction
    - bad leakage checks
    """

    answer = agent_state.get("final_answer") or ""
    answer_lower = answer.lower()

    expected_supplier_id = eval_case.get("expected_supplier_id")
    expected_supplier_name = eval_case.get("expected_supplier_name")
    expected_risk_band = eval_case.get("expected_risk_band")
    expected_terms = eval_case.get("expected_terms") or []

    expects_supplier_resolution = bool(eval_case.get("expects_supplier_resolution"))
    expects_assumption_correction = bool(eval_case.get("expects_assumption_correction"))
    expects_evidence = bool(eval_case.get("expects_evidence"))

    actual_supplier_id = agent_state.get("supplier_id")
    actual_supplier_name = agent_state.get("supplier_name")

    evidence = agent_state.get("cleaned_evidence_result", {}).get("evidence", []) or []

    # Supplier resolution score
    if expects_supplier_resolution:
        supplier_resolution_correct = int(
            actual_supplier_id == expected_supplier_id
            and actual_supplier_name == expected_supplier_name
        )
    else:
        supplier_resolution_correct = int(actual_supplier_id is None)

    # Risk band check
    if expected_risk_band is None:
        risk_band_correct = 1
    else:
        risk_band_correct = int(str(expected_risk_band).lower() in answer_lower)

    # Expected term coverage
    matched_terms = []
    missing_terms = []

    for term in expected_terms:
        if term is None:
            continue

        term_text = str(term).strip()

        if len(term_text) == 0:
            continue

        if term_text.lower() in answer_lower:
            matched_terms.append(term_text)
        else:
            missing_terms.append(term_text)

    expected_term_coverage = (
        len(matched_terms) / max(len(matched_terms) + len(missing_terms), 1)
    )

    # Evidence checks
    if expects_evidence:
        evidence_present = int(len(evidence) > 0)
    else:
        evidence_present = int(len(evidence) == 0)

    # Assumption correction
    if expects_assumption_correction:
        assumption_correction_pass = int(
            "not high risk" in answer_lower
            or "not classified as high" in answer_lower
            or "classified as medium" in answer_lower
        )
    else:
        assumption_correction_pass = 1

    # Leakage checks
    none_leakage_flag = int(
        "recommended action is: none" in answer_lower
        or "recommendation:\nnone" in answer_lower
        or "top risk driver is none" in answer_lower
    )

    raw_payload_leakage_flag = int(
        "<?xml" in answer_lower
        or "<sdnlist" in answer_lower
        or '"raw_text"' in answer_lower
    )

    unresolved_expected_pass = 1
    if not expects_supplier_resolution:
        unresolved_expected_pass = int(
            actual_supplier_id is None
            and (
                "could not identify" in answer_lower
                or "include a supplier id" in answer_lower
                or "specific supplier" in answer_lower
            )
        )

    # Overall quality score
    component_scores = [
        supplier_resolution_correct,
        risk_band_correct,
        expected_term_coverage,
        evidence_present,
        assumption_correction_pass,
        1 - none_leakage_flag,
        1 - raw_payload_leakage_flag,
        unresolved_expected_pass
    ]

    quality_score = sum(component_scores) / len(component_scores)

    return {
        "supplier_resolution_correct": int(supplier_resolution_correct),
        "risk_band_correct": int(risk_band_correct),
        "expected_term_coverage": float(expected_term_coverage),
        "matched_terms": matched_terms,
        "missing_terms": missing_terms,
        "evidence_present": int(evidence_present),
        "assumption_correction_pass": int(assumption_correction_pass),
        "none_leakage_flag": int(none_leakage_flag),
        "raw_payload_leakage_flag": int(raw_payload_leakage_flag),
        "unresolved_expected_pass": int(unresolved_expected_pass),
        "quality_score": float(quality_score)
    }


batch_eval_results = []

batch_started_at = datetime.now(timezone.utc)

for idx, eval_case in enumerate(agent_eval_questions, start=1):
    print(f"\nRunning eval {idx}/{len(agent_eval_questions)}: {eval_case['eval_id']}")
    print("Question:", eval_case["question"])

    try:
        agent_state = run_supplier_risk_agent_with_mlflow(eval_case["question"])
        monitoring_metrics = extract_agent_monitoring_metrics(agent_state)
        eval_scores = score_agent_response(eval_case, agent_state)

        result_row = {
            "eval_id": eval_case["eval_id"],
            "eval_category": eval_case["eval_category"],
            "question": eval_case["question"],
            "expected_supplier_id": eval_case.get("expected_supplier_id"),
            "expected_supplier_name": eval_case.get("expected_supplier_name"),
            "expected_risk_band": eval_case.get("expected_risk_band"),
            "expects_supplier_resolution": bool(eval_case.get("expects_supplier_resolution")),
            "expects_assumption_correction": bool(eval_case.get("expects_assumption_correction")),
            "expects_evidence": bool(eval_case.get("expects_evidence")),
            "actual_ok": bool(agent_state.get("ok")),
            "actual_error": agent_state.get("error"),
            "actual_supplier_id": agent_state.get("supplier_id"),
            "actual_supplier_name": agent_state.get("supplier_name"),
            "actual_risk_band": agent_state.get("risk_band"),
            "actual_recommended_action": agent_state.get("recommended_action"),
            "mlflow_run_id": agent_state.get("mlflow_run_id"),
            "duration_seconds": float(agent_state.get("duration_seconds") or 0.0),
            "wrapper_runtime_seconds": float(agent_state.get("wrapper_runtime_seconds") or 0.0),
            "trace_step_count": int(monitoring_metrics.get("trace_step_count") or 0),
            "evidence_count": int(monitoring_metrics.get("evidence_count") or 0),
            "retrieval_candidate_count": int(monitoring_metrics.get("retrieval_candidate_count") or 0),
            "retrieval_returned_count": int(monitoring_metrics.get("retrieval_returned_count") or 0),
            "unique_source_count": int(monitoring_metrics.get("unique_source_count") or 0),
            "answer_char_length": int(monitoring_metrics.get("answer_char_length") or 0),
            "supplier_resolution_correct": int(eval_scores["supplier_resolution_correct"]),
            "risk_band_correct": int(eval_scores["risk_band_correct"]),
            "expected_term_coverage": float(eval_scores["expected_term_coverage"]),
            "evidence_present": int(eval_scores["evidence_present"]),
            "assumption_correction_pass": int(eval_scores["assumption_correction_pass"]),
            "none_leakage_flag": int(eval_scores["none_leakage_flag"]),
            "raw_payload_leakage_flag": int(eval_scores["raw_payload_leakage_flag"]),
            "unresolved_expected_pass": int(eval_scores["unresolved_expected_pass"]),
            "quality_score": float(eval_scores["quality_score"]),
            "matched_terms_json": json.dumps(eval_scores["matched_terms"], ensure_ascii=False),
            "missing_terms_json": json.dumps(eval_scores["missing_terms"], ensure_ascii=False),
            "final_answer": agent_state.get("final_answer"),
            "eval_run_status": "success",
            "eval_error": None,
            "evaluated_at": datetime.now(timezone.utc).isoformat()
        }

        print(
            f"Result: ok={result_row['actual_ok']} | "
            f"supplier={result_row['actual_supplier_id']} | "
            f"quality={result_row['quality_score']:.3f} | "
            f"mlflow_run_id={result_row['mlflow_run_id']}"
        )

    except Exception as e:
        result_row = {
            "eval_id": eval_case["eval_id"],
            "eval_category": eval_case["eval_category"],
            "question": eval_case["question"],
            "expected_supplier_id": eval_case.get("expected_supplier_id"),
            "expected_supplier_name": eval_case.get("expected_supplier_name"),
            "expected_risk_band": eval_case.get("expected_risk_band"),
            "expects_supplier_resolution": bool(eval_case.get("expects_supplier_resolution")),
            "expects_assumption_correction": bool(eval_case.get("expects_assumption_correction")),
            "expects_evidence": bool(eval_case.get("expects_evidence")),
            "actual_ok": False,
            "actual_error": str(e),
            "actual_supplier_id": None,
            "actual_supplier_name": None,
            "actual_risk_band": None,
            "actual_recommended_action": None,
            "mlflow_run_id": None,
            "duration_seconds": 0.0,
            "wrapper_runtime_seconds": 0.0,
            "trace_step_count": 0,
            "evidence_count": 0,
            "retrieval_candidate_count": 0,
            "retrieval_returned_count": 0,
            "unique_source_count": 0,
            "answer_char_length": 0,
            "supplier_resolution_correct": 0,
            "risk_band_correct": 0,
            "expected_term_coverage": 0.0,
            "evidence_present": 0,
            "assumption_correction_pass": 0,
            "none_leakage_flag": 0,
            "raw_payload_leakage_flag": 0,
            "unresolved_expected_pass": 0,
            "quality_score": 0.0,
            "matched_terms_json": "[]",
            "missing_terms_json": "[]",
            "final_answer": None,
            "eval_run_status": "failed",
            "eval_error": str(e),
            "evaluated_at": datetime.now(timezone.utc).isoformat()
        }

        print("Eval failed:", str(e))

    batch_eval_results.append(result_row)

batch_finished_at = datetime.now(timezone.utc)

batch_eval_results_pd = pd.DataFrame(batch_eval_results)

print("\nBatch evaluation complete.")
print("Started:", batch_started_at.isoformat())
print("Finished:", batch_finished_at.isoformat())
print("Eval rows:", len(batch_eval_results_pd))
print("Mean quality score:", round(batch_eval_results_pd["quality_score"].mean(), 3))
print("Success count:", int(batch_eval_results_pd["actual_ok"].sum()))

display(batch_eval_results_pd[
    [
        "eval_id",
        "eval_category",
        "actual_ok",
        "actual_supplier_id",
        "expected_supplier_id",
        "supplier_resolution_correct",
        "risk_band_correct",
        "expected_term_coverage",
        "evidence_present",
        "assumption_correction_pass",
        "none_leakage_flag",
        "raw_payload_leakage_flag",
        "quality_score",
        "mlflow_run_id"
    ]
])

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 28 — Cell 7
# Save batch evaluation results to Delta
# ─────────────────────────────────────────────────────────────

import json
import pandas as pd
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    BooleanType,
    TimestampType
)

assert "batch_eval_results_pd" in globals(), "batch_eval_results_pd missing. Rerun Cell 6."
assert len(batch_eval_results_pd) > 0, "batch_eval_results_pd is empty."
assert "AGENT_EVAL_TABLE" in globals(), "AGENT_EVAL_TABLE missing. Rerun Cell 2A."

eval_batch_id = f"eval_batch_{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}"

eval_rows = []

for row in batch_eval_results_pd.to_dict(orient="records"):
    eval_rows.append({
        "eval_batch_id": eval_batch_id,
        "eval_question_set_version": EVAL_QUESTION_SET_VERSION,
        "eval_id": row.get("eval_id"),
        "eval_category": row.get("eval_category"),
        "question": row.get("question"),
        "expected_supplier_id": row.get("expected_supplier_id"),
        "expected_supplier_name": row.get("expected_supplier_name"),
        "expected_risk_band": row.get("expected_risk_band"),
        "expects_supplier_resolution": bool(row.get("expects_supplier_resolution")),
        "expects_assumption_correction": bool(row.get("expects_assumption_correction")),
        "expects_evidence": bool(row.get("expects_evidence")),
        "actual_ok": bool(row.get("actual_ok")),
        "actual_error": row.get("actual_error"),
        "actual_supplier_id": row.get("actual_supplier_id"),
        "actual_supplier_name": row.get("actual_supplier_name"),
        "actual_risk_band": row.get("actual_risk_band"),
        "actual_recommended_action": row.get("actual_recommended_action"),
        "mlflow_run_id": row.get("mlflow_run_id"),
        "duration_seconds": float(row.get("duration_seconds") or 0.0),
        "wrapper_runtime_seconds": float(row.get("wrapper_runtime_seconds") or 0.0),
        "trace_step_count": int(row.get("trace_step_count") or 0),
        "evidence_count": int(row.get("evidence_count") or 0),
        "retrieval_candidate_count": int(row.get("retrieval_candidate_count") or 0),
        "retrieval_returned_count": int(row.get("retrieval_returned_count") or 0),
        "unique_source_count": int(row.get("unique_source_count") or 0),
        "answer_char_length": int(row.get("answer_char_length") or 0),
        "supplier_resolution_correct": int(row.get("supplier_resolution_correct") or 0),
        "risk_band_correct": int(row.get("risk_band_correct") or 0),
        "expected_term_coverage": float(row.get("expected_term_coverage") or 0.0),
        "evidence_present": int(row.get("evidence_present") or 0),
        "assumption_correction_pass": int(row.get("assumption_correction_pass") or 0),
        "none_leakage_flag": int(row.get("none_leakage_flag") or 0),
        "raw_payload_leakage_flag": int(row.get("raw_payload_leakage_flag") or 0),
        "unresolved_expected_pass": int(row.get("unresolved_expected_pass") or 0),
        "quality_score": float(row.get("quality_score") or 0.0),
        "matched_terms_json": row.get("matched_terms_json"),
        "missing_terms_json": row.get("missing_terms_json"),
        "final_answer": row.get("final_answer"),
        "eval_run_status": row.get("eval_run_status"),
        "eval_error": row.get("eval_error"),
        "evaluated_at": row.get("evaluated_at"),
        "saved_at": datetime.now(timezone.utc)
    })

eval_schema = StructType([
    StructField("eval_batch_id", StringType(), False),
    StructField("eval_question_set_version", StringType(), True),
    StructField("eval_id", StringType(), True),
    StructField("eval_category", StringType(), True),
    StructField("question", StringType(), True),
    StructField("expected_supplier_id", StringType(), True),
    StructField("expected_supplier_name", StringType(), True),
    StructField("expected_risk_band", StringType(), True),
    StructField("expects_supplier_resolution", BooleanType(), True),
    StructField("expects_assumption_correction", BooleanType(), True),
    StructField("expects_evidence", BooleanType(), True),
    StructField("actual_ok", BooleanType(), True),
    StructField("actual_error", StringType(), True),
    StructField("actual_supplier_id", StringType(), True),
    StructField("actual_supplier_name", StringType(), True),
    StructField("actual_risk_band", StringType(), True),
    StructField("actual_recommended_action", StringType(), True),
    StructField("mlflow_run_id", StringType(), True),
    StructField("duration_seconds", DoubleType(), True),
    StructField("wrapper_runtime_seconds", DoubleType(), True),
    StructField("trace_step_count", IntegerType(), True),
    StructField("evidence_count", IntegerType(), True),
    StructField("retrieval_candidate_count", IntegerType(), True),
    StructField("retrieval_returned_count", IntegerType(), True),
    StructField("unique_source_count", IntegerType(), True),
    StructField("answer_char_length", IntegerType(), True),
    StructField("supplier_resolution_correct", IntegerType(), True),
    StructField("risk_band_correct", IntegerType(), True),
    StructField("expected_term_coverage", DoubleType(), True),
    StructField("evidence_present", IntegerType(), True),
    StructField("assumption_correction_pass", IntegerType(), True),
    StructField("none_leakage_flag", IntegerType(), True),
    StructField("raw_payload_leakage_flag", IntegerType(), True),
    StructField("unresolved_expected_pass", IntegerType(), True),
    StructField("quality_score", DoubleType(), True),
    StructField("matched_terms_json", StringType(), True),
    StructField("missing_terms_json", StringType(), True),
    StructField("final_answer", StringType(), True),
    StructField("eval_run_status", StringType(), True),
    StructField("eval_error", StringType(), True),
    StructField("evaluated_at", StringType(), True),
    StructField("saved_at", TimestampType(), True),
])

eval_results_df = spark.createDataFrame(eval_rows, schema=eval_schema)

(
    eval_results_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit("28_mlflow_agent_monitoring_evaluation"))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(AGENT_EVAL_TABLE)
)

print(f"Saved agent evaluation results to: {AGENT_EVAL_TABLE}")
print("eval_batch_id:", eval_batch_id)

display(
    spark.table(AGENT_EVAL_TABLE)
    .filter(F.col("eval_batch_id") == eval_batch_id)
    .select(
        "eval_id",
        "eval_category",
        "actual_ok",
        "actual_supplier_id",
        "expected_supplier_id",
        "supplier_resolution_correct",
        "risk_band_correct",
        "expected_term_coverage",
        "evidence_present",
        "assumption_correction_pass",
        "none_leakage_flag",
        "raw_payload_leakage_flag",
        "quality_score",
        "mlflow_run_id"
    )
    .orderBy("eval_id")
)

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 28 — Cell 8
# Create agent monitoring summary view/table for dashboard/UI
# ─────────────────────────────────────────────────────────────

from pyspark.sql import functions as F

AGENT_MONITORING_SUMMARY_TABLE = "supplysage_gold.gold_supplier_risk_agent_monitoring_summary"

monitoring_df = spark.table(AGENT_MONITORING_TABLE)
eval_df = spark.table(AGENT_EVAL_TABLE)

# Latest monitoring metrics
monitoring_summary = (
    monitoring_df
    .groupBy("agent_name", "agent_version")
    .agg(
        F.count("*").alias("total_monitoring_runs"),
        F.sum(F.when(F.col("ok") == True, 1).otherwise(0)).alias("successful_monitoring_runs"),
        F.avg("duration_seconds").alias("avg_agent_duration_seconds"),
        F.avg("wrapper_runtime_seconds").alias("avg_wrapper_runtime_seconds"),
        F.avg("evidence_count").alias("avg_evidence_count"),
        F.avg("unique_source_count").alias("avg_unique_source_count"),
        F.sum("none_leakage_flag").alias("none_leakage_count"),
        F.sum("raw_payload_leakage_flag").alias("raw_payload_leakage_count"),
        F.max("monitoring_logged_at").alias("latest_monitoring_logged_at")
    )
)

# Latest evaluation metrics
eval_summary = (
    eval_df
    .groupBy("eval_question_set_version")
    .agg(
        F.count("*").alias("total_eval_rows"),
        F.countDistinct("eval_batch_id").alias("eval_batch_count"),
        F.avg("quality_score").alias("avg_quality_score"),
        F.avg("supplier_resolution_correct").alias("supplier_resolution_accuracy"),
        F.avg("risk_band_correct").alias("risk_band_accuracy"),
        F.avg("expected_term_coverage").alias("avg_expected_term_coverage"),
        F.avg("evidence_present").alias("evidence_presence_rate"),
        F.avg("assumption_correction_pass").alias("assumption_correction_rate"),
        F.sum("none_leakage_flag").alias("eval_none_leakage_count"),
        F.sum("raw_payload_leakage_flag").alias("eval_raw_payload_leakage_count"),
        F.max("saved_at").alias("latest_eval_saved_at")
    )
    .withColumn("join_key", F.lit(1))
)

monitoring_summary = monitoring_summary.withColumn("join_key", F.lit(1))

summary_df = (
    monitoring_summary
    .join(eval_summary, on="join_key", how="full")
    .drop("join_key")
    .withColumn("summary_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit("28_mlflow_agent_monitoring_evaluation"))
)

(
    summary_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(AGENT_MONITORING_SUMMARY_TABLE)
)

print(f"Saved monitoring summary to: {AGENT_MONITORING_SUMMARY_TABLE}")

display(
    spark.table(AGENT_MONITORING_SUMMARY_TABLE)
)